# 02 — Preparação e engenharia de atributos / Data preparation and feature engineering

**PT:** Este notebook transforma as bases brutas em tabelas numéricas consistentes para modelagem. Ele gera arquivos intermediários em `DATA/processed/`.

**EN:** This notebook transforms the raw datasets into consistent numeric tables for modeling. It generates intermediate files in `DATA/processed/`.

## Decisões / Decisions

- **PT:** `Name`, `Ticket` e `Cabin` são removidos nesta versão por terem alta cardinalidade e exigirem análise específica. `Sex` e `Embarked` são codificados; idade, tarifa e porto ausentes são imputados com estatísticas do treino.
- **EN:** `Name`, `Ticket`, and `Cabin` are removed in this version because of their high cardinality and need for dedicated analysis. `Sex` and `Embarked` are encoded; missing age, fare, and port values are imputed with training-set statistics.

- **PT:** `FamilySize` e `IsAlone` resumem a composição familiar. A normalização fica para o pipeline de modelagem, onde é aprendida somente no subconjunto de treino.
- **EN:** `FamilySize` and `IsAlone` summarize family composition. Scaling is left to the modeling pipeline, where it is fitted only on the training split.

In [ ]:
from pathlib import Path

import pandas as pd

DATA_DIR = Path('../DATA')
OUTPUT_DIR = DATA_DIR / 'processed'
OUTPUT_DIR.mkdir(exist_ok=True)

train_raw = pd.read_csv(DATA_DIR / 'train.csv')
test_raw = pd.read_csv(DATA_DIR / 'test.csv')

## Função de preparação / Preparation function

**PT:** A mesma função é aplicada às duas bases, com estatísticas aprendidas no treino. Isso mantém colunas e regras alinhadas.

**EN:** The same function is applied to both datasets using statistics learned from training data. This keeps columns and rules aligned.

In [ ]:
DROP_COLUMNS = ['Name', 'Ticket', 'Cabin']

def prepare_features(data, age_median, fare_median, embarked_mode):
    prepared = data.drop(columns=DROP_COLUMNS).copy()
    prepared['Age'] = prepared['Age'].fillna(age_median)
    prepared['Fare'] = prepared['Fare'].fillna(fare_median)
    prepared['Embarked'] = prepared['Embarked'].fillna(embarked_mode)
    prepared['FamilySize'] = prepared['SibSp'] + prepared['Parch']
    prepared['IsAlone'] = (prepared['FamilySize'] == 0).astype(int)
    prepared['Sex'] = (prepared['Sex'] == 'male').astype(int)
    prepared = pd.get_dummies(prepared, columns=['Embarked'], dtype=int)
    return prepared

age_median = train_raw['Age'].median()
fare_median = train_raw['Fare'].median()
embarked_mode = train_raw['Embarked'].mode()[0]

train_prepared = prepare_features(train_raw, age_median, fare_median, embarked_mode)
test_prepared = prepare_features(test_raw, age_median, fare_median, embarked_mode)

# Ensures that train and test carry exactly the same predictors.
feature_columns = train_prepared.drop(columns='Survived').columns
test_prepared = test_prepared.reindex(columns=feature_columns, fill_value=0)

## Verificações / Checks

**PT:** Confirmamos que não há valores ausentes e que treino e teste usam os mesmos atributos preditores.

**EN:** We confirm that no missing values remain and that training and test use the same predictor columns.

In [ ]:
assert train_prepared.isna().sum().sum() == 0
assert test_prepared.isna().sum().sum() == 0
assert list(train_prepared.drop(columns='Survived').columns) == list(test_prepared.columns)

print(f'Treino preparado / Prepared training: {train_prepared.shape}')
print(f'Teste preparado / Prepared test: {test_prepared.shape}')
train_prepared.head()

## Salvar dados preparados / Save prepared data

**PT:** Os arquivos abaixo são entradas explícitas dos próximos notebooks e podem ser recriados executando esta etapa.

**EN:** The files below are explicit inputs for the next notebooks and can be recreated by running this step.

In [ ]:
train_prepared.to_csv(OUTPUT_DIR / 'train_prepared.csv', index=False)
test_prepared.to_csv(OUTPUT_DIR / 'test_prepared.csv', index=False)

print(f'Arquivos salvos em / Files saved to: {OUTPUT_DIR.resolve()}')